In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
import random
import pickle
from copy import deepcopy
from typing import List

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import datasets, transforms
from sklearn.metrics import confusion_matrix
import torch.nn.functional as F

SEED = 42
DATA_ROOT = "./data"
NUM_WORKERS = 0
MOMENTUM = 0.9
WEIGHT_DECAY = 5e-4

SAVE_DIR = "/content/drive/MyDrive/ML_Project/project_files/GridSearch_Mahalanobis"
os.makedirs(SAVE_DIR, exist_ok=True)

GN_GROUPS = 32
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# -----------------------------
# PyTorch 2.6 compat loader (fix UnpicklingError / weights_only default)
# -----------------------------
def torch_load_compat(path, map_location="cpu"):
    """
    PyTorch 2.6 default changed to weights_only=True.
    Your checkpoints include metadata / numpy scalars -> can fail.
    Use weights_only=False ONLY if you trust the checkpoint source (your Drive).
    """
    return torch.load(path, map_location=map_location, weights_only=False)

def get_best_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

device = get_best_device()
PIN_MEM = (device.type == "cuda")
if device.type == "cuda":

    print(f"[INFO] Using device: {device}")
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True)
# -----------------------------
# paths
# -----------------------------
MD_DIR = SAVE_DIR
os.makedirs(MD_DIR, exist_ok=True)

# Task1 stats (0,1)
MD_TASK1_STATS_PATH = os.path.join(MD_DIR, "best_epoch_mahalanobis_stats.pth")

# Task2 weights (model after learning 2,3) => 4-class head
MD_TASK2_WEIGHTS_PATH = os.path.join(MD_DIR, "finetuned_task2_best.pth")

# Task2 stats (2,3)
TASK2_BEST_STATS_PATH = os.path.join(MD_DIR, "task2_best_epoch_qda_stats_classes2,3.pth")

# Fisher
TOPK_PATH      = os.path.join(MD_DIR, "CS_class0-3_topk.pkl")
NEIGHBORS_PATH = os.path.join(MD_DIR, "CS_neighbors_class0-3.pkl")

# Save Task3 best weights + stats (4,5)
FINAL_BEST_WEIGHTS = os.path.join(MD_DIR, "finetuned_task3_best.pth")
TASK3_BEST_STATS_PATH = os.path.join(MD_DIR, "task3_best_epoch_qda_stats_classes4,5.pth")

# -----------------------------
# GroupNorm / ResNet-18
# -----------------------------
def make_gn(C: int) -> nn.GroupNorm:
    g = min(GN_GROUPS, C)
    while g > 1 and (C % g) != 0:
        g //= 2
    return nn.GroupNorm(num_groups=max(1, g), num_channels=C)

def conv3x3(in_planes, out_planes, stride=1):
    return nn.Conv2d(in_planes, out_planes, kernel_size=3, stride=stride, padding=1, bias=False)

class BasicBlock(nn.Module):
    expansion = 1
    def __init__(self, in_planes, planes, stride=1):
        super().__init__()
        self.conv1 = conv3x3(in_planes, planes, stride)
        self.gn1 = make_gn(planes)
        self.conv2 = conv3x3(planes, planes)
        self.gn2 = make_gn(planes)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, planes, kernel_size=1, stride=stride, bias=False),
                make_gn(planes)
            )

    def forward(self, x):
        out = torch.relu(self.gn1(self.conv1(x)))
        out = self.gn2(self.conv2(out))
        out += self.shortcut(x)
        return torch.relu(out)

class ResNet18Backbone(nn.Module):
    def __init__(self, nf=64):
        super().__init__()
        self.nf = nf
        self.conv1 = conv3x3(3, nf)
        self.gn1 = make_gn(nf)
        self.layer1 = nn.Sequential(BasicBlock(nf, nf), BasicBlock(nf, nf))
        self.layer2 = nn.Sequential(BasicBlock(nf, nf*2, 2), BasicBlock(nf*2, nf*2))
        self.layer3 = nn.Sequential(BasicBlock(nf*2, nf*4, 2), BasicBlock(nf*4, nf*4))
        self.layer4 = nn.Sequential(BasicBlock(nf*4, nf*8, 2), BasicBlock(nf*8, nf*8))

    def forward(self, x):
        x = torch.relu(self.gn1(self.conv1(x)))
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = F.avg_pool2d(x, x.shape[2])
        return x.view(x.size(0), -1)

    @property
    def out_dim(self):
        return self.nf * 8

class SingleHeadNet(nn.Module):
    def __init__(self, backbone, num_classes):
        super().__init__()
        self.backbone = backbone
        self.head = nn.Linear(backbone.out_dim, num_classes)

    def forward(self, x):
        return self.head(self.backbone(x))

# -----------------------------
# dataset
# -----------------------------
def get_cifar10_datasets():
    tf_train = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.4914,0.4822,0.4465),(0.2470,0.2435,0.2616))
    ])
    tf_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914,0.4822,0.4465),(0.2470,0.2435,0.2616))
    ])
    train = datasets.CIFAR10(root=DATA_ROOT, train=True, download=True, transform=tf_train)
    test  = datasets.CIFAR10(root=DATA_ROOT, train=False, download=True, transform=tf_test)
    return train, test

def indices_for_classes(dataset, keep):
    t = dataset.targets if hasattr(dataset, "targets") else [dataset[i][1] for i in range(len(dataset))]
    return [i for i,y in enumerate(t) if int(y) in keep]

class RemapDataset(Dataset):
    def __init__(self, dataset, indices, keep_classes):
        self.dataset = dataset
        self.indices = indices
        self.mapping = {c:i for i,c in enumerate(sorted(keep_classes))}
    def __len__(self): return len(self.indices)
    def __getitem__(self, idx):
        x,y = self.dataset[self.indices[idx]]
        return x,y

def make_loader(ds, bs, shuffle):
    return DataLoader(ds, batch_size=bs, shuffle=shuffle, num_workers=NUM_WORKERS, pin_memory=PIN_MEM)

# -----------------------------
# Load Fisher data
# -----------------------------
def load_fisher_data():
    with open(TOPK_PATH, "rb") as f: topk = pickle.load(f)
    with open(NEIGHBORS_PATH, "rb") as f: neigh = pickle.load(f)
    print(f"[INFO] Loaded Fisher data | TopK={len(topk)} | Neigh={len(neigh)}")
    return topk, neigh

# -----------------------------
#  EWC + Freeze
# -----------------------------
def build_neighbor_tensors(model, fisher_neighbors, neighbor_original_values):
    pm = dict(model.named_parameters())
    usable = [n for n in fisher_neighbors if n['name'] in pm]
    if not usable: return {}
    max_f = max((n['cs'] for n in usable), default=1.0) or 1.0
    buckets = {}
    for n in usable:
        buckets.setdefault(n['name'], []).append((int(n['index']), float(n['cs'])/max_f))
    ewc = {}
    for name, lst in buckets.items():
        lst.sort(key=lambda t:t[0])
        idxs = torch.tensor([i for i,_ in lst], device=device, dtype=torch.long)
        fish = torch.tensor([f for _,f in lst], device=device, dtype=torch.float32)
        flat = pm[name].view(-1)
        orig = torch.stack([neighbor_original_values[(name,int(i))] for i in idxs.tolist()]).to(flat.device, dtype=flat.dtype)
        ewc[name] = {'idxs': idxs, 'fish': fish, 'orig': orig}
    return ewc

def build_freeze_masks_and_cache(model, topk_list):
    masks, frozen_idxs, frozen_vals = {}, {}, {}
    pm = dict(model.named_parameters())
    by_name = {}

    for e in topk_list:
        n, i = e['name'], int(e['index'])
        if (n in pm) and (not n.startswith("head")):
            by_name.setdefault(n, []).append(i)

    for name, idxs in by_name.items():
        p = pm[name]
        flat = p.detach().view(-1)
        idxs_t = torch.tensor(idxs, device=flat.device, dtype=torch.long)

        m = torch.ones_like(p, dtype=torch.bool, device=p.device)
        mv = m.view(-1)
        mv[idxs_t] = False

        masks[name] = mv.view_as(p)
        frozen_idxs[name] = idxs_t
        frozen_vals[name] = flat.index_select(0, idxs_t).clone()

    return masks, frozen_idxs, frozen_vals

def apply_freeze_after_backward(model, masks):
    with torch.no_grad():
        for n,p in model.named_parameters():
            m = masks.get(n,None)
            if p.grad is not None and m is not None:
                p.grad.mul_(m.to(p.grad.dtype))

@torch.no_grad()
def apply_strict_freeze_after_step(param_map, frozen_idxs, frozen_vals):
    for n,idxs in frozen_idxs.items():
        if n in param_map:
            flat = param_map[n].view(-1)
            flat.index_copy_(0, idxs, frozen_vals[n].to(flat.device, dtype=flat.dtype))

# -----------------------------
# QDA stats + prediction
# -----------------------------
@torch.no_grad()
def compute_md_stats_from_loader(backbone: nn.Module, loader, class_ids: List[int]):
    backbone.eval()
    feats = []
    labels = []

    for x, y in loader:
        x = x.to(device)
        f = backbone(x).detach().cpu()
        feats.append(f)
        labels.append(y.detach().cpu())

    feats = torch.cat(feats, dim=0)    # [N,D]
    labels = torch.cat(labels, dim=0)  # [N]

    means, inv_covs, logdets = [], [], []

    for c in class_ids:
        cf = feats[labels == c]
        if cf.numel() == 0:
            raise RuntimeError(f"No samples found for class {c} to compute QDA stats.")

        mean = cf.mean(0)
        centered = cf - mean

        cov = torch.cov(centered.T)
        eps = 1e-5
        cov = cov + eps * torch.eye(cov.size(0))

        sign, logdet = torch.slogdet(cov)
        if sign.item() <= 0:
            cov = cov + 1e-3 * torch.eye(cov.size(0))
            sign, logdet = torch.slogdet(cov)

        inv = torch.inverse(cov)

        means.append(mean)
        inv_covs.append(inv)
        logdets.append(logdet)

    means = torch.stack(means, dim=0).to(device)       # [K,D]
    inv_covs = torch.stack(inv_covs, dim=0).to(device) # [K,D,D]
    logdets = torch.stack(logdets, dim=0).to(device)   # [K]
    return means, inv_covs, logdets

@torch.no_grad()
def md_scores(backbone: nn.Module, x, means, inv_covs, logdets):
    backbone.eval()
    f = backbone(x)  # [B,D]
    K = means.size(0)
    scores = []
    for k in range(K):
        diff = f - means[k]
        quad = torch.sum((diff @ inv_covs[k]) * diff, dim=1)
        score = quad + logdets[k]
        scores.append(score.unsqueeze(1))
    return torch.cat(scores, dim=1)  # [B,K]

# --------- NEW: full QDA discriminant ----------
@torch.no_grad()
def qda_scores(backbone: nn.Module, x, means, inv_covs, logdets, priors):
    """
    g_k(x) = -0.5 * quad - 0.5 * logdet + log(pi_k)
    returns [B, K] (bigger = more likely)
    """
    backbone.eval()
    f = backbone(x)  # [B, D]
    scores = []
    K = means.size(0)
    for k in range(K):
        diff = f - means[k]  # [B, D]
        quad = torch.sum((diff @ inv_covs[k]) * diff, dim=1)  # [B]
        gk = -0.5 * quad - 0.5 * logdets[k] + torch.log(priors[k] + 1e-20)
        scores.append(gk.unsqueeze(1))
    return torch.cat(scores, dim=1)  # [B, K]

# --------- UPDATED: eval 6-way uses qda_scores and priors ----------
@torch.no_grad()
def eval_loader_qda_6way(model, loader,
                         m01, inv01, ld01, p01,
                         m23, inv23, ld23, p23,
                         m45, inv45, ld45, p45):
    model.eval()
    preds, labels = [], []

    for x, y in loader:
        x = x.to(device)
        y = y.to(device)

        s01 = qda_scores(model.backbone, x, m01, inv01, ld01, p01)  # [B,2]
        s23 = qda_scores(model.backbone, x, m23, inv23, ld23, p23)  # [B,2]
        s45 = qda_scores(model.backbone, x, m45, inv45, ld45, p45)  # [B,2]

        s_all = torch.cat([s01, s23, s45], dim=1)  # [B,6]
        pred = torch.argmax(s_all, dim=1)          # choose highest g_k

        preds.extend(pred.detach().cpu().numpy())
        labels.extend(y.detach().cpu().numpy())

    labels_np = np.array(labels)
    preds_np = np.array(preds)

    acc = 100.0 * np.mean(preds_np == labels_np)
    cm = confusion_matrix(labels_np, preds_np, labels=[0,1,2,3,4,5])
    return acc, cm

# -----------------------------
#  loading stats + PyTorch 2.6 fix
# -----------------------------
def load_qda_stats_file(path, expected_classes):
    st = torch_load_compat(path, map_location="cpu")  # use compat loader

    classes = st.get("classes", expected_classes)
    assert list(classes) == list(expected_classes), f"Expected classes {expected_classes}, got {classes}"

    means = st["means"].to(device)
    inv   = st["inv_covs"].to(device)

    if "logdets" in st:
        logdet = st["logdets"].to(device)
    else:
        logdet_list = []
        for k in range(inv.size(0)):
            sign, ld = torch.slogdet(inv[k])
            logdet_list.append(-ld)
        logdet = torch.stack(logdet_list).to(device)

    # NEW: load priors (fallback to uniform if missing)
    priors = st.get("priors", None)
    if priors is None:
        priors = torch.ones(inv.size(0), device=device) / float(inv.size(0))
    else:
        priors = priors.to(device)

    return means, inv, logdet, priors

# -----------------------------
# init model for Task3 (6 classes) starting from Task2 (4-class weights)
# -----------------------------
def init_expanded_model_task3():
    ckpt = torch_load_compat(MD_TASK2_WEIGHTS_PATH, map_location=device)
    state4 = ckpt["state"] if isinstance(ckpt, dict) and "state" in ckpt else ckpt

    backbone4 = ResNet18Backbone(64)
    model4 = SingleHeadNet(backbone4, 4).to(device)
    model4.load_state_dict(state4, strict=True)

    backbone6 = ResNet18Backbone(64)
    model6 = SingleHeadNet(backbone6, 6).to(device)

    model6.backbone.load_state_dict(model4.backbone.state_dict(), strict=True)

    # copy first 4 rows of head
    with torch.no_grad():
        model6.head.weight[:4, :] = model4.head.weight[:4, :]
        model6.head.bias[:4] = model4.head.bias[:4]

    return model6

# -----------------------------
# Helper: compute mean cosine (feature <-> correct head weight) for a loader
def collect_cosine_mean(model, W, loader):
    model.eval()
    device_loc = next(model.parameters()).device
    cos_accum = []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device_loc)
            y = y.to(device_loc)
            feats = model.backbone(x)             # [B, D]
            w = W[y]                              # [B, D]
            cosine = F.cosine_similarity(feats, w, dim=1)  # [B]
            cos_accum.append(cosine.cpu())
    if len(cos_accum) == 0:
        return float('nan')
    return torch.cat(cos_accum).mean().item()
@torch.no_grad()
def normalize_weights_l2(model, eps=1e-12):
    for p in model.parameters():
        if not p.requires_grad:
            continue

        if p.ndim >= 2:
            w = p.view(p.size(0), -1)
            norms = w.norm(2, dim=1, keepdim=True).clamp_min(eps)
            w.div_(norms)
        else:
            # Usually skip bias
            pass
# -----------------------------
# Training one config for Task3: train on (4,5), evaluate 6-way QDA
#  - Use FULL logits for CrossEntropy (labels are 4 or 5)
#  - IMPORTANT: head rows 0..3 are NOT frozen anymore (head is trainable)
# -----------------------------
def train_one_config_task3(lr_head, lr_backbone, bs, epochs, lambda_ewc, topk_fisher, fisher_neighbors):
    model = init_expanded_model_task3()
        # ------------------ freeze head rows 0..3 (register hooks + keep original copy) ------------------
    n_old = 4  # freeze classes 0,1,2,3

    # get references
    W = model.head.weight   # shape [6, feat_dim]
    B = model.head.bias     # shape [6]

    # build masks (float on params)
    weight_mask = torch.ones_like(W, device=W.device)
    bias_mask   = torch.ones_like(B, device=B.device)

    if n_old > 0:
        weight_mask[:n_old, :] = 0.0
        bias_mask[:n_old]      = 0.0

    # keep original values for safety restore after optimizer.step()
    head_orig = {
        "weight": W.data[:n_old, :].clone(),
        "bias":   B.data[:n_old].clone()
    }

    # hook functions (zero gradients on frozen rows)
    def _freeze_weight_grad(grad):
        return grad * weight_mask.to(grad.device)

    def _freeze_bias_grad(grad):
        return grad * bias_mask.to(grad.device)

    # register hooks once (model is local to this call)
    W.register_hook(_freeze_weight_grad)
    B.register_hook(_freeze_bias_grad)
    # ----------------------------------------------------------------------------------------------
    param_map = {n:p for n,p in model.named_parameters()}
    masks, frozen_idxs, frozen_vals = build_freeze_masks_and_cache(model, topk_fisher)

    # old stats for 0,1 and 2,3 (now include priors)
    m01, inv01, ld01, p01 = load_qda_stats_file(MD_TASK1_STATS_PATH, [0,1])
    m23, inv23, ld23, p23 = load_qda_stats_file(TASK2_BEST_STATS_PATH, [2,3])

    with torch.no_grad():
        cpu_cache = {n:p.view(-1).detach().cpu() for n,p in model.named_parameters()}
    neighbor_original_values = {}
    for n in fisher_neighbors:
        name, idx = n['name'], int(n['index'])
        if name in cpu_cache and idx < cpu_cache[name].numel():
            neighbor_original_values[(name,idx)] = cpu_cache[name][idx]
    ewc_tensors = build_neighbor_tensors(model, fisher_neighbors, neighbor_original_values)

    optimizer = optim.SGD([
        {"params": model.backbone.parameters(), "lr": lr_backbone, "weight_decay":WEIGHT_DECAY},
        {"params": model.head.parameters(), "lr": lr_head, "weight_decay":0.0}
    ], momentum=MOMENTUM)

    train_set, test_set = get_cifar10_datasets()

    # Task3 train on 4,5
    train_45 = RemapDataset(train_set, indices_for_classes(train_set,[4,5]), [4,5])

    # tests
    test_01 = RemapDataset(test_set, indices_for_classes(test_set,[0,1]), [0,1])
    test_23 = RemapDataset(test_set, indices_for_classes(test_set,[2,3]), [2,3])
    test_45 = RemapDataset(test_set, indices_for_classes(test_set,[4,5]), [4,5])
    test_all = RemapDataset(test_set, indices_for_classes(test_set,[0,1,2,3,4,5]), [0,1,2,3,4,5])

    train_loader = make_loader(train_45, bs, True)
    train_45_eval_loader = make_loader(train_45, 256, False)

    test_01_loader = make_loader(test_01, 256, False)
    test_23_loader = make_loader(test_23, 256, False)
    test_45_loader = make_loader(test_45, 256, False)
    test_all_loader = make_loader(test_all, 256, False)

    best_avg, best_epoch = 0, 0
    best_state, best_results = None, None

    best_m45 = None
    best_inv45 = None
    best_ld45 = None
    best_p45 = None

    for e in range(1, epochs+1):
        model.train(); loss_sum=0; correct=0; total=0

        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad(set_to_none=True)

            logits = model(imgs)  # [B,6]

            # EWC penalty
            ewc_penalty = 0.0
            if lambda_ewc != 0 and len(ewc_tensors) > 0:
                for name, pack in ewc_tensors.items():
                    p = param_map[name].view(-1)
                    diff = p.index_select(0, pack['idxs']) - pack['orig']
                    ewc_penalty += (pack['fish'] * (diff**2)).sum()

            # ---- use FULL logits for CrossEntropy (labels are 4 or 5) ----
            loss_ce = F.cross_entropy(logits, labels)   # logits shape [B,6], labels in {4,5}
            # -------------------------------------------------------------------

            loss = loss_ce + (lambda_ewc/2.0) * ewc_penalty
            loss.backward()

            apply_freeze_after_backward(model, masks)  # only masks from top-k (non-head)
            optimizer.step()
            normalize_weights_l2(model)
            with torch.no_grad():
                if n_old > 0:
                    W.data[:n_old, :] = head_orig["weight"]
                    B.data[:n_old]     = head_orig["bias"]



            # ensure strictly frozen top-k params unchanged (head not among them since we didn't add it)
            param_map_after = {n:p.data for n,p in model.named_parameters()}
            apply_strict_freeze_after_step(param_map_after, frozen_idxs, frozen_vals)

            # compute training accuracy on the (4,5) examples using logits[:,4:6]
            logits_new = logits[:, 4:6]   # classes 4,5
            labels_remap = labels - 4     # 4->0, 5->1
            preds_task = logits_new.argmax(1)
            correct += (preds_task == labels_remap).sum().item()
            total += labels_remap.size(0)
            loss_sum += float(loss.detach().cpu())

        acc_train = 100.0 * correct / max(1,total)

        m45, inv45, ld45 = compute_md_stats_from_loader(model.backbone, train_45_eval_loader, class_ids=[4,5])

        count4 = 0
        count5 = 0
        for _, y_batch in train_45_eval_loader:
            ys = y_batch.detach().cpu().numpy()
            count4 += int((ys == 4).sum())
            count5 += int((ys == 5).sum())

        total_45 = float(count4 + count5)
        if total_45 <= 0:
            p45 = torch.tensor([0.5, 0.5], device=device)
        else:
            p45 = torch.tensor([count4 / total_45, count5 / total_45], device=device)
        # -------------------------------------------------------------------------

        acc01, _ = eval_loader_qda_6way(model, test_01_loader,
                                       m01, inv01, ld01, p01,
                                       m23, inv23, ld23, p23,
                                       m45, inv45, ld45, p45)
        acc23, _ = eval_loader_qda_6way(model, test_23_loader,
                                       m01, inv01, ld01, p01,
                                       m23, inv23, ld23, p23,
                                       m45, inv45, ld45, p45)
        acc45, _ = eval_loader_qda_6way(model, test_45_loader,
                                       m01, inv01, ld01, p01,
                                       m23, inv23, ld23, p23,
                                       m45, inv45, ld45, p45)
        acc_all, cm_all = eval_loader_qda_6way(model, test_all_loader,
                                              m01, inv01, ld01, p01,
                                              m23, inv23, ld23, p23,
                                              m45, inv45, ld45, p45)

        avg_acc = (acc01 + acc23 + acc45) / 3.0

        print(f"Epoch {e:03d} | Loss={loss_sum/len(train_loader):.4f} | Train(4,5)={acc_train:.2f}% | "
              f"01(QDA)={acc01:.2f}% | 23(QDA)={acc23:.2f}% | 45(QDA)={acc45:.2f}% | Unified6(QDA)={acc_all:.2f}% | Avg={avg_acc:.2f}%")

        if avg_acc > best_avg:
            best_avg = avg_acc
            best_epoch = e
            best_state = {k:v.cpu() for k,v in model.state_dict().items()}
            best_results = (acc01, acc23, acc45, acc_all, avg_acc)

            best_m45 = m45.detach().cpu()
            best_inv45 = inv45.detach().cpu()
            best_ld45 = ld45.detach().cpu()
            best_p45 = p45.detach().cpu()

    print(f"[INFO] Best Epoch: {best_epoch} | AvgAcc={best_results[4]:.2f}% | "
          f"01={best_results[0]:.2f}% | 23={best_results[1]:.2f}% | 45={best_results[2]:.2f}% | Unified6={best_results[3]:.2f}%")

    if best_m45 is None:
        raise RuntimeError("No best epoch found; training may have failed or epochs=0.")
    # best_p45 should be set when best epoch updated
    if best_p45 is None:
        # fallback: compute from last p45 (if exists) or uniform
        try:
            best_p45 = p45.detach().cpu()
        except Exception:
            best_p45 = torch.ones(2) / 2.0

    torch.save({
        "means": best_m45,
        "inv_covs": best_inv45,
        "logdets": best_ld45,
        "priors": best_p45,
        "classes": [4,5],
        "best_epoch": best_epoch,
        "avg_acc": best_avg,
        "results": {
            "acc_01": best_results[0],
            "acc_23": best_results[1],
            "acc_45": best_results[2],
            "acc_unified6": best_results[3],
        }
    }, TASK3_BEST_STATS_PATH)
    print(f"[INFO] Saved Task3 QDA stats (classes 4,5) -> {TASK3_BEST_STATS_PATH}")

    return best_avg, best_state

# -----------------------------
# Grid Search Task3
# -----------------------------
def grid_search_finetune_task3():
    topk_fisher, fisher_neighbors = load_fisher_data()

    LR_HEADS = [0.1,0.2, 0.3, 0.5, 0.7]
    LR_BACKBONES = [1e-5]
    LAMBDAS = [2]
    BATCH_SIZES = [64,128,256]
    EPOCHS_LIST = [20]

    best_acc, best_weights, best_cfg = -1, None, None

    for lr_h in LR_HEADS:
        for lr_b in LR_BACKBONES:
            for lam in LAMBDAS:
                for bs in BATCH_SIZES:
                    for ep in EPOCHS_LIST:
                        print("\n" + "="*70)
                        print(f"🚀 Task3 Config | lr_head={lr_h} | lr_backbone={lr_b} | λ={lam} | bs={bs} | epochs={ep}")
                        print("="*70)

                        acc, weights = train_one_config_task3(
                            lr_head=lr_h, lr_backbone=lr_b,
                            bs=bs, epochs=ep, lambda_ewc=lam,
                            topk_fisher=topk_fisher, fisher_neighbors=fisher_neighbors
                        )

                        if acc > best_acc:
                            best_acc = acc
                            best_weights = deepcopy(weights)
                            best_cfg = {"lr_head":lr_h,"lr_backbone":lr_b,"lambda":lam,"batch_size":bs,"epochs":ep}

    print("\n" + "#"*70)
    print(f"[DONE] Task3 Best AvgAcc={best_acc:.2f}% | Config={best_cfg}")
    print("#"*70)

    if best_weights is not None:
        torch.save(best_weights, FINAL_BEST_WEIGHTS)
        print(f"[INFO] Saved Task3 best weights -> {FINAL_BEST_WEIGHTS}")
    else:
        print("[WARN] No best weights to save.")

# -----------------------------
# Main
# -----------------------------
if __name__ == "__main__":
    grid_search_finetune_task3()